# GardnAIze — 04 Explore and validate garden images

Objective: build an auditable labeled dataset for **garden**, **not_garden**, **unusable**. This notebook does not download or redistribute images. Confirm each image's license and provenance before use.

Expected layout: `datasets/images/garden_validation/{garden,not_garden,unusable}/<images>`; manually label images; do not infer soil type from appearance.

In [2]:
import pandas as pd
from PIL import Image, UnidentifiedImageError
from pathlib import Path
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA = ROOT / "datasets" / "images" / "garden_validation"
CLASSES = ["garden", "not_garden", "unusable"]
print("Dataset:", DATA)

records = []
for label in CLASSES:
    for path in sorted((DATA / label).glob("*")):
        if not path.is_file():
            continue
        try:
            with Image.open(path) as im:
                im.verify()
            with Image.open(path) as im:
                width, height = im.size
            records.append({"path":str(path),"label":label,"width":width,"height":height,"valid":True})
        except (UnidentifiedImageError, OSError, ValueError):
            records.append({"path":str(path),"label":label,"width":None,"height":None,"valid":False})
df = pd.DataFrame(records,columns=["path","label","width","height","valid"])
display(df.groupby(["label","valid"]).size().rename("images").reset_index()) if not df.empty else print("No labeled images found. Add licensed images first.")

Dataset: /home/bsui/projects/GardnAIze/datasets/images/garden_validation


,label,valid,images
0,garden,True,28
1,not_garden,True,24
2,unusable,True,24


## Provenance and leakage controls

Maintain a separate local CSV with source URL, author, license, retrieval date, image hash, labeler and group identifier (same garden / same photographer). Split by group to prevent near-duplicate leakage. Do not use plant species API data as garden-scene image labels.

In [3]:
import hashlib
if not df.empty:
    df["sha256"] = df["path"].map(lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest())
    print("Duplicate image hashes:", int(df["sha256"].duplicated().sum()))
    display(df.head())

Duplicate image hashes: 0


,path,label,width,height,valid,sha256
0,/home/bsui/projects/GardnAIze/datasets/images/...,garden,2048,1152,True,47aec6a30fad5c8080f759ba0e6cec75c9b862532edcdb...
1,/home/bsui/projects/GardnAIze/datasets/images/...,garden,1333,1000,True,26a040a1aaee82c0daaa0bbf147d49789490b448c9781d...
2,/home/bsui/projects/GardnAIze/datasets/images/...,garden,1024,572,True,6a242c18b709335bd64fc0c61effb114c5627a31c1698c...
3,/home/bsui/projects/GardnAIze/datasets/images/...,garden,1024,682,True,7b1eb8705a3f5f97cce49ace4b82ab22b4452d692fb127...
4,/home/bsui/projects/GardnAIze/datasets/images/...,garden,603,838,True,e6fbca83d4ce97642b42a5f64968530bd975437bebc7a9...


## Data provenance and leakage audit

Verify image provenance, duplicate files and source-group overlap
before training the classifier.

In [6]:
from pathlib import Path
import pandas as pd

# Locate the project root
cwd = Path.cwd().resolve()
ROOT = next(
    (
        p for p in [cwd, *cwd.parents]
        if (p / "datasets").is_dir()
        and (p / "notebooks").is_dir()
    ),
    None,
)

if ROOT is None:
    raise RuntimeError("GardnAIze project root not found")

manifest_path = ROOT / "datasets/images/garden_image_provenance.csv"
manifest = pd.read_csv(manifest_path).fillna("")

# Columns currently available in the CSV
required = {
    "file",
    "label",
    "commons_page",
    "sha256",
    "review_status",
}

missing = required - set(manifest.columns)

if missing:
    raise ValueError(f"Missing manifest columns: {sorted(missing)}")

# Derive split from image directory
split_map = {
    "garden_train": "train",
    "garden_validation": "validation",
    "garden_test": "test",
}

def get_split(file_path):
    parts = Path(file_path).parts
    matches = [
        split_name
        for folder, split_name in split_map.items()
        if folder in parts
    ]
    return matches[0] if len(matches) == 1 else "unknown"

manifest["split"] = manifest["file"].apply(get_split)

# Check files
manifest["exists"] = manifest["file"].apply(
    lambda p: (ROOT / p).is_file()
)

print("Total images:", len(manifest))

print("\nImages by split:")
print(manifest["split"].value_counts())

print("\nImages by label:")
print(manifest["label"].value_counts())

print("\nReview status:")
print(manifest["review_status"].value_counts())

print("\nMissing files:", (~manifest["exists"]).sum())

# Check exact duplicates across splits
hash_splits = (
    manifest[manifest["sha256"].astype(str).str.strip() != ""]
    .groupby("sha256")["split"]
    .nunique()
)

duplicate_hashes = hash_splits[hash_splits > 1]

print("Duplicate hashes across splits:", len(duplicate_hashes))

# Check repeated Wikimedia source pages
page_splits = (
    manifest[manifest["commons_page"].astype(str).str.strip() != ""]
    .groupby("commons_page")["split"]
    .nunique()
)

duplicate_pages = page_splits[page_splits > 1]

print("Commons pages across splits:", len(duplicate_pages))

# Detect potential related images using source titles
manifest["source_prefix"] = (
    manifest["source_title"]
    .astype(str)
    .str.replace(r"\([^)]*\)", "", regex=True)
    .str.replace(r"[_\-]+", " ", regex=True)
    .str.strip()
    .str.lower()
    .str[:45]
)

related = (
    manifest[manifest["source_prefix"] != ""]
    .groupby("source_prefix")
    .agg(
        count=("file", "size"),
        n_splits=("split", "nunique"),
        splits=("split", lambda x: ", ".join(sorted(set(x)))),
    )
)

suspects = related[
    (related["count"] > 1)
    & (related["n_splits"] > 1)
]

print("\nPotential related-image groups across splits:", len(suspects))
display(suspects.head(30))

Total images: 84

Images by split:
split
validation    72
test          12
Name: count, dtype: int64

Images by label:
label
garden        30
not_garden    30
unusable      24
Name: count, dtype: int64

Review status:
review_status
pending_manual_review                    60
synthetic_derived_from_training_image    24
Name: count, dtype: int64

Missing files: 0
Duplicate hashes across splits: 0
Commons pages across splits: 0

Potential related-image groups across splits: 1


,count,n_splits,splits
source_prefix,,,
file:rma urban roots st paul ag in the city .,21,2,"test, validation"


In [7]:
suspect_rows = manifest[
    manifest["source_prefix"]
    == "file:rma urban roots st paul ag in the city ."
].copy()

display(
    suspect_rows[
        ["file", "label", "split", "source_title", "review_status"]
    ].sort_values(["split", "file"])
)

print("\nDistribution by split:")
display(
    pd.crosstab(
        suspect_rows["split"],
        suspect_rows["label"]
    )
)

,file,label,split,source_title,review_status
0,datasets/images/garden_test/garden/garden_0001...,garden,test,File:RMA-Urban Roots St Paul-Ag in the city (2...,pending_manual_review
5,datasets/images/garden_test/garden/garden_0006...,garden,test,File:RMA-Urban Roots St Paul-Ag in the city (2...,pending_manual_review
10,datasets/images/garden_test/garden/garden_0011...,garden,test,File:RMA-Urban Roots St Paul-Ag in the city (2...,pending_manual_review
15,datasets/images/garden_test/garden/garden_0016...,garden,test,File:RMA-Urban Roots St Paul-Ag in the city (2...,pending_manual_review
20,datasets/images/garden_test/garden/garden_0021...,garden,test,File:RMA-Urban Roots St Paul-Ag in the city (2...,pending_manual_review
1,datasets/images/garden_validation/garden/garde...,garden,validation,File:RMA-Urban Roots St Paul-Ag in the city (2...,pending_manual_review
3,datasets/images/garden_validation/garden/garde...,garden,validation,File:RMA-Urban Roots St Paul-Ag in the city (2...,pending_manual_review
7,datasets/images/garden_validation/garden/garde...,garden,validation,File:RMA-Urban Roots St Paul-Ag in the city (2...,pending_manual_review
13,datasets/images/garden_validation/garden/garde...,garden,validation,File:RMA-Urban Roots St Paul-Ag in the city (2...,pending_manual_review
16,datasets/images/garden_validation/garden/garde...,garden,validation,File:RMA-Urban Roots St Paul-Ag in the city (2...,pending_manual_review



Distribution by split:


label,garden,unusable
split,,
test,5,0
validation,8,8


## Data provenance and leakage audit

Check file integrity, image provenance, annotation status
and potential overlap between dataset splits.

In [ ]:
from pathlib import Path
from collections import Counter
import pandas as pd

# Locate project root
cwd = Path.cwd().resolve()
ROOT = next(
    (p for p in [cwd, *cwd.parents]
     if (p / "datasets").is_dir() and (p / "notebooks").is_dir()),
    None,
)

if ROOT is None:
    raise RuntimeError("GardnAIze project root not found")

manifest_path = ROOT / "datasets/images/garden_image_provenance.csv"
df = pd.read_csv(manifest_path).fillna("")

required = {
    "file", "label", "commons_page",
    "sha256", "source_title", "review_status",
}

missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing columns: {sorted(missing)}")

# Derive split from the actual file path
split_map = {
    "garden_train": "train",
    "garden_validation": "validation",
    "garden_test": "test",
}

def get_split(path):
    parts = Path(path).parts
    matches = [value for folder, value in split_map.items() if folder in parts]
    return matches[0] if len(matches) == 1 else "unknown"

df["split"] = df["file"].apply(get_split)

# Check referenced files
df["exists"] = df["file"].apply(lambda p: (ROOT / p).is_file())

print("Total manifest entries:", len(df))
print("\nSplit distribution:")
print(df["split"].value_counts())

print("\nLabels:")
print(df["label"].value_counts())

print("\nManual review status:")
print(df["review_status"].value_counts())

print("\nMissing files:", (~df["exists"]).sum())
print("Unknown splits:", (df["split"] == "unknown").sum())

# Detect identical files assigned to different splits
valid_hashes = df[df["sha256"].astype(str).str.strip() != ""]
hash_splits = valid_hashes.groupby("sha256")["split"].nunique()
duplicate_leaks = hash_splits[hash_splits > 1]

print("\nExact duplicate hashes across splits:", len(duplicate_leaks))

if not duplicate_leaks.empty:
    display(
        df[df["sha256"].isin(duplicate_leaks.index)][
            ["file", "label", "split", "sha256"]
        ].sort_values("sha256")
    )

# Detect identical Commons source pages across splits
valid_pages = df[df["commons_page"].astype(str).str.strip() != ""]
page_splits = valid_pages.groupby("commons_page")["split"].nunique()
page_leaks = page_splits[page_splits > 1]

print("Identical Commons pages across splits:", len(page_leaks))

if not page_leaks.empty:
    display(
        df[df["commons_page"].isin(page_leaks.index)][
            ["file", "label", "split", "commons_page"]
        ].sort_values("commons_page")
    )

# Potentially related images: same source title prefix
df["source_prefix"] = (
    df["source_title"]
    .str.replace(r"\([^)]*\)", "", regex=True)
    .str.replace(r"[_\-]+", " ", regex=True)
    .str.strip()
    .str.lower()
    .str[:45]
)

related = (
    df[df["source_prefix"] != ""]
    .groupby("source_prefix")
    .agg(
        count=("file", "size"),
        splits=("split", lambda s: ", ".join(sorted(set(s)))),
        n_splits=("split", "nunique"),
    )
)

suspects = related[
    (related["count"] > 1) &
    (related["n_splits"] > 1)
]

print("\nPotential related-image groups across splits:", len(suspects))
display(suspects.head(30))

if (
    (~df["exists"]).any()
    or (df["split"] == "unknown").any()
    or not duplicate_leaks.empty
    or not page_leaks.empty
):
    print("\nAUDIT: Issues detected — inspect before training.")
else:
    print("\nAUDIT: No missing files or exact cross-split duplicates detected.")

print("Related-image groups require manual review.")

Next: review class balance, label ambiguity and image licenses before training. `unusable` needs examples of blur, extreme darkness and low resolution, not simply non-garden photos.